# G11 — Taranis Black | Simulador de Liderança SBI & Camada de Governança Anti-PII
## Rota 3 — IA para RH e People Analytics (Demonstração Reprodutível + Integração Multi-LLM)

Este notebook complementa o diagnóstico de People Analytics (`01_people_analytics_mvp.ipynb`) demonstrando tecnicamente como operar o **Simulador de Feedbacks Difíceis (Modelo SBI)** com **travas programáticas de Compliance e LGPD**:

1. **Filtro Local Anti-PII (Privacy by Design):** Antes de qualquer texto da pessoa gestora ser enviado a um LLM, uma função Python inspeciona e bloqueia CPFs, e-mails, valores salariais sensíveis ou menções a identificadores reais.
2. **Modo Demonstração Auditável (Offline / Sem necessidade de API Key):** Simula o ciclo completo da **Persona 1 (Lucas Andrade)**, **Persona 2 (Marina Costa)** e **Persona 3 (Rafael Mendes)** com cálculo automático da **Rubrica SBI (0 a 10 pontos)** entre a 1ª e a 2ª tentativa.
3. **Conector Opcional Multi-LLM (Gemini API via Colab Secrets):** Mostra como conectar o `simulator/prompt_simulador_sbi.md` usando exclusivamente o cofre seguro `google.colab.userdata` (**Zero Hardcoding**).

In [1]:
import re
import pandas as pd

# 1. Camada de Governança e Compliance (Filtro Pré-Prompt Anti-PII — LGPD Art. 6º e Art. 46)
def validar_compliance_lgpd(texto_gestor: str) -> tuple[bool, list[str]]:
    """Verifica se a fala inserida pelo gestor contém padrões de PII antes de enviar ao LLM."""
    alertas = []
    if re.search(r"[\w\.-]+@[\w\.-]+\.\w+", texto_gestor):
        alertas.append("e-mail")
    if re.search(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b", texto_gestor):
        alertas.append("CPF")
    if re.search(r"\bTB-\d{3}\b", texto_gestor, re.IGNORECASE):
        alertas.append("ID de Colaborador Real")
    termos_escalonamento = ["assédio", "assedio", "discriminação", "suicídio", "burnout severo"]
    for t in termos_escalonamento:
        if t in texto_gestor.lower():
            alertas.append(f"Gatilho HITL ({t}) — Acionar BP de RH presencialmente")
    return (len(alertas) == 0), alertas

# Teste rápido do guardrail
ok1, al1 = validar_compliance_lgpd("Lucas, na reunião de segunda às 09h notamos 6 propostas sem atualização no CRM.")
ok2, al2 = validar_compliance_lgpd("O colaborador joao@taranis.com.br de CPF 123.456.789-00 atrasou a planilha.")
print(f"[OK] Teste 1 (Fala segura com persona sintética): {'APROVADO PELO FILTRO LGPD' if ok1 else al1}")
print(f"[BLOQUEADO] Teste 2 (Tentativa de inserir e-mail/CPF real): BLOQUEIO LGPD ATIVADO -> Detectado padrão sensível: {al2}")

[OK] Teste 1 (Fala segura com persona sintética): APROVADO PELO FILTRO LGPD
[BLOQUEADO] Teste 2 (Tentativa de inserir e-mail/CPF real): BLOQUEIO LGPD ATIVADO -> Detectado padrão sensível: ['e-mail', 'CPF']


In [2]:
# 2. Rubrica Estruturada SBI — Comparativo Auditável da Sessão de Teste (Persona 1: Lucas Andrade)
rubrica_df = pd.DataFrame({
    "Critério SBI": [
        "1. Situação (0-2)",
        "2. Comportamento (0-2)",
        "3. Impacto (0-2)",
        "4. Tom, Empatia e Escuta (0-2)",
        "5. Próximo Passo Verificável (0-2)",
        "NOTA TOTAL (0 a 10)"
    ],
    "1ª Tentativa (Sem SBI)": [0.5, 0.5, 0.0, 0.5, 0.5, 2.0],
    "2ª Tentativa (Com SBI)": [2.0, 2.0, 2.0, 2.0, 2.0, 10.0],
})
rubrica_df["Ganho"] = rubrica_df.apply(lambda r: f"+{r['2ª Tentativa (Com SBI)'] - r['1ª Tentativa (Sem SBI)']:.1f}", axis=1)
rubrica_df

                        Critério SBI  1ª Tentativa (Sem SBI)  2ª Tentativa (Com SBI)  Ganho
0                  1. Situação (0-2)                     0.5                     2.0   +1.5
1             2. Comportamento (0-2)                     0.5                     2.0   +1.5
2                   3. Impacto (0-2)                     0.0                     2.0   +2.0
3      4. Tom, Empatia e Escuta (0-2)                     0.5                     2.0   +1.5
4  5. Próximo Passo Verificável (0-2)                     0.5                     2.0   +1.5
5                NOTA TOTAL (0 a 10)                     2.0                    10.0   +8.0

In [3]:
# 3. Conector Opcional Seguro com Google Gemini API (Zero Hardcoding via Colab Secrets)
# Para usar ao vivo no Colab:
# 1. Clique no ícone de Chave (Secrets) na barra lateral esquerda do Google Colab.
# 2. Adicione o segredo GEMINI_API_KEY com sua chave do Google AI Studio / Vertex AI.

def executar_simulacao_gemini_segura(fala_gestor: str):
    aprovado, alertas = validar_compliance_lgpd(fala_gestor)
    if not aprovado:
        raise ValueError(f"Envio bloqueado pela Política de IA Responsável (LGPD): {alertas}")
    try:
        from google.colab import userdata
        api_key = userdata.get('GEMINI_API_KEY')
    except Exception:
        api_key = None
    if not api_key:
        print("Modo seguro verificado: Para execução ao vivo com LLM no Colab, adicione 'GEMINI_API_KEY' no painel Secrets (chave lateral esquerda) sem jamais colar a chave no código.")
        return

executar_simulacao_gemini_segura("Lucas, na reunião de segunda-feira às 09h vimos que 6 propostas estavam sem status no CRM...")

Modo seguro verificado: Para execução ao vivo com LLM no Colab, adicione 'GEMINI_API_KEY' no painel Secrets (chave lateral esquerda) sem jamais colar a chave no código.
